# Data Exploration Traffic Dataset

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [5]:
# Load the dataset

data = np.load('data/dataset.npz')
X_train = data["X_train"]  # [1261, 36, 48]
Y_train = data["Y_train"]  # [1261, 36]
X_test = data["X_test"]  # [840, 36, 48]
Y_test = data["Y_test"]  # [840, 36]
adj_mat = data["adj_mat"]  # [36, 36]

[[[0.09248015 0.09668379 0.07006072 ... 0.         0.         0.        ]
  [0.09715086 0.10228865 0.0817375  ... 0.         0.         0.        ]
  [0.11536665 0.11069594 0.10555815 ... 0.         0.         0.        ]
  ...
  [0.06772536 0.0817375  0.06118636 ... 0.         0.         1.        ]
  [0.09388136 0.11069594 0.08827651 ... 0.         0.         1.        ]
  [0.08313872 0.06679122 0.06725829 ... 0.         1.         0.        ]]

 [[0.09668379 0.07006072 0.06585708 ... 0.         0.         0.        ]
  [0.10228865 0.0817375  0.07659972 ... 0.         0.         0.        ]
  [0.11069594 0.10555815 0.09528258 ... 0.         0.         0.        ]
  ...
  [0.0817375  0.06118636 0.045773   ... 0.         0.         1.        ]
  [0.11069594 0.08827651 0.06492294 ... 0.         0.         1.        ]
  [0.06679122 0.06725829 0.05324615 ... 0.         1.         0.        ]]

 [[0.07006072 0.06585708 0.078468   ... 0.         0.         0.        ]
  [0.0817375  0.076599

## Understanding the column order
As there are no labels in the dataset, it is hard to tell with 100% accuracy which columns represents which.
I'm sure the first 10 columns are the traffic volume.
But then, things get blurry.

- 0-9: historical sequence, 10 most recent data points
- 10-16: Week day
- ERROR: 17-41: Hours of day -> Column 39: Weird output, not binary, but integre with values of 5

In [6]:
# Flatten
X_flat = X_train.reshape(-1, X_train.shape[-1])  # [1261*36, 48]

# Variance per feature
variances = X_flat.var(axis=0)

df = pd.DataFrame({
    "column": np.arange(X_train.shape[-1]),
    "variance": variances
})

print(df)

    column  variance
0        0  0.039828
1        1  0.039846
2        2  0.039861
3        3  0.039869
4        4  0.039868
5        5  0.039860
6        6  0.039847
7        7  0.039830
8        8  0.039823
9        9  0.039829
10      10  0.176278
11      11  0.176270
12      12  0.176262
13      13  0.136091
14      14  0.129032
15      15  0.042425
16      16  0.042425
17      17  0.042425
18      18  0.040241
19      19  0.039543
20      20  0.039543
21      21  0.039543
22      22  0.039543
23      23  0.039543
24      24  0.039543
25      25  0.039543
26      26  0.039543
27      27  0.039543
28      28  0.039543
29      29  0.039543
30      30  0.039543
31      31  0.039543
32      32  0.039543
33      33  0.039543
34      34  0.039543
35      35  0.039543
36      36  0.039543
37      37  0.039543
38      38  0.039543
39      39  0.693629
40      40  0.172815
41      41  0.237670
42      42  0.098789
43      43  0.200583
44      44  0.243098
45      45  0.212214
46      46  0

In [7]:
for col in range(X_train.shape[-1]):
    unique_vals = np.unique(X_train[..., col])
    if len(unique_vals) <= 3:
        print(f"Column {col}: {unique_vals}")

# Just states that columns 0-9 and 39 are not binary. The rest is binary.

Column 10: [0. 1.]
Column 11: [0. 1.]
Column 12: [0. 1.]
Column 13: [0. 1.]
Column 14: [0. 1.]
Column 15: [0. 1.]
Column 16: [0. 1.]
Column 17: [0. 1.]
Column 18: [0. 1.]
Column 19: [0. 1.]
Column 20: [0. 1.]
Column 21: [0. 1.]
Column 22: [0. 1.]
Column 23: [0. 1.]
Column 24: [0. 1.]
Column 25: [0. 1.]
Column 26: [0. 1.]
Column 27: [0. 1.]
Column 28: [0. 1.]
Column 29: [0. 1.]
Column 30: [0. 1.]
Column 31: [0. 1.]
Column 32: [0. 1.]
Column 33: [0. 1.]
Column 34: [0. 1.]
Column 35: [0. 1.]
Column 36: [0. 1.]
Column 37: [0. 1.]
Column 38: [0. 1.]
Column 40: [0. 1.]
Column 41: [0. 1.]
Column 42: [0. 1.]
Column 43: [0. 1.]
Column 44: [0. 1.]
Column 45: [0. 1.]
Column 46: [0. 1.]
Column 47: [0. 1.]


In [9]:
from collections import defaultdict

X = X_train.reshape(-1, X_train.shape[-1])  # [45396, 48]

binary_cols = []
for col in range(X.shape[1]):
    vals = np.unique(X[:, col])
    if set(vals).issubset({0, 1}):
        binary_cols.append(col)

groups = []

remaining = set(binary_cols)

while remaining:
    col = remaining.pop()
    group = {col}

    for other_col in list(remaining):
        candidate_group = sorted(group | {other_col})

        # Check if at most one column is active per row
        row_sums = X[:, candidate_group].sum(axis=1)

        if np.all(row_sums <= 1):
            group.add(other_col)
            remaining.remove(other_col)

    groups.append(sorted(group))

print("Mutually exclusive binary groups:")

for g in groups:
    row_sums = X[:, g].sum(axis=1)

    if np.all(row_sums == 1):
        status = "complete one-hot"
    elif np.all(row_sums <= 1):
        status = "mutually exclusive, but not always active"
    else:
        status = "not one-hot"

    print(f"{g}: {status}")

Mutually exclusive binary groups:
[10, 11, 12, 13, 14]: complete one-hot
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]: complete one-hot
[40, 41, 42, 43]: complete one-hot
[44, 45, 46, 47]: complete one-hot


In [11]:
blocks = {
    "weekday_candidate": range(10, 15),
    "hour_candidate": range(15, 39),
    "unknown_col_39": [39],
    "direction_candidate": range(40, 44),
    "road_candidate": range(44, 48),
}

X = X_train.reshape(-1, X_train.shape[-1])

for name, cols in blocks.items():
    cols = list(cols)
    sums = X[:, cols].sum(axis=1)

    print(name, cols)
    print("unique row sums:", np.unique(sums))
    print()

weekday_candidate [10, 11, 12, 13, 14]
unique row sums: [1.]

hour_candidate [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]
unique row sums: [1.]

unknown_col_39 [39]
unique row sums: [1. 2. 3. 4. 5.]



KeyboardInterrupt: 